In [15]:
import pandas as pd
import re
import unicodedata

In [16]:
df1 = pd.read_excel('../data/processed/eventos_jujuy_limpio.xlsx')

In [17]:
df2 = pd.read_excel("../data/processed/eventos_chubut_limpio.xlsx")

In [18]:
df_j = df1.copy()
df_ch = df2.copy()

In [19]:
df_j.columns

Index(['ID', 'Localidad', 'Localidad_detalle', 'Actores participantes',
       'Demanda principal', 'Tipo de acción', 'Tipo de acción_detalle',
       'Disrupción', 'Escala', 'Intensidad', 'Respuesta estatal',
       'Tipo Int. Estatal', 'Presencia Policial', 'Otros datos', 'Uso Fuerza',
       'Heridos', 'Detenciones', 'Judicialización', 'Coord. Interinst.',
       'Canales Inform.', 'Amenaza Política', 'Resultado', 'Fuente',
       'Observaciones', 'Notas_estandarizacion', '_actores_norm', 'Docentes',
       'Gremios_estatales', 'Sindicatos_privados', 'Organizaciones_sociales',
       'Comunidades_indigenas', 'Vecinos', 'Colectivos_especificos',
       'Partidos_politicos', 'Publico_general', 'demanda_salarial',
       'demanda_laboral', 'demanda_ambiental_territorial',
       'demanda_politica_legal', 'demanda_ddhh', 'demanda_servicios',
       'demanda_derechos_especificos', 'demanda_institucional',
       'demanda_conmemoracion'],
      dtype='str')

In [20]:
df_ch.columns

Index(['noticia_id', 'link', 'fecha_publicacion', 'titulo', 'localidad',
       'actores_participantes', 'demanda_principal', 'tipo_accion',
       'tipo_accion_detalle', 'disrupcion', 'escala', 'intensidad',
       'respuesta_estatal', 'presencia_policial', 'accionar_policial',
       'tipo_intervencion_estatal', 'uso_fuerza', 'heridos', 'detenciones',
       'judicializacion', 'coord_interinstitucional', 'canales_informales',
       'amenaza_politica', 'resultado_episodio', 'fuente', 'observaciones',
       'notas_estandarizacion', '_actores_norm', 'docentes',
       'gremios_estatales', 'sindicatos_privados', 'organizaciones_sociales',
       'comunidades_indigenas', 'vecinos', 'colectivos_ambientales',
       'colectivos_especificos', 'partidos_politicos', 'policias_actores',
       'publico_general', 'colectivos_violentos', 'otros', 'demanda_salarial',
       'demanda_laboral', 'demanda_ambiental_territorial',
       'demanda_politica_legal', 'demanda_ddhh', 'demanda_servicios',
 

In [21]:
# ============================================
# 1. RENOMBRAR JUJUY a snake_case
# ============================================
RENOMBRAR_JUJUY = {
    "ID": "noticia_id",
    "Localidad": "localidad",
    "Actores participantes": "actores_participantes",
    "Demanda principal": "demanda_principal",
    "Tipo de acción": "tipo_accion",
    "Tipo de acción_detalle": "tipo_accion_detalle",
    "Disrupción": "disrupcion",
    "Escala": "escala",
    "Intensidad": "intensidad",
    "Respuesta estatal": "respuesta_estatal",
    "Tipo Int. Estatal": "tipo_intervencion_estatal",
    "Presencia Policial": "presencia_policial",
    "Uso Fuerza": "uso_fuerza",
    "Heridos": "heridos",
    "Detenciones": "detenciones",
    "Judicialización": "judicializacion",
    "Coord. Interinst.": "coord_interinstitucional",
    "Canales Inform.": "canales_informales",
    "Amenaza Política": "amenaza_politica",
    "Resultado": "resultado_episodio",
    "Fuente": "fuente",
    "Observaciones": "observaciones",
    "Notas_estandarizacion": "notas_estandarizacion",
    # Actores
    "Docentes": "docentes",
    "Gremios_estatales": "gremios_estatales",
    "Sindicatos_privados": "sindicatos_privados",
    "Organizaciones_sociales": "organizaciones_sociales",
    "Comunidades_indigenas": "comunidades_indigenas",
    "Vecinos": "vecinos",
    "Colectivos_especificos": "colectivos_especificos",
    "Partidos_politicos": "partidos_politicos",
    "Publico_general": "publico_general",
}

df_j = df_j.rename(columns=RENOMBRAR_JUJUY)

# ============================================
# 2. DROPEAR columnas que no van
# ============================================
DROP_CHUBUT = ["link", "fecha_publicacion", "titulo",
               "actores_norm", "actores_limpio", "_actores_norm"]
DROP_JUJUY = ["Localidad_detalle", "Otros datos", "_actores_norm"]

df_chubut = df_ch.drop(columns=[c for c in DROP_CHUBUT if c in df_ch.columns])
df_jujuy = df_j.drop(columns=[c for c in DROP_JUJUY if c in df_j.columns])

# ============================================
# 3. AGREGAR columnas faltantes
# ============================================
COLS_ACTORES = [
    "docentes", "gremios_estatales", "sindicatos_privados",
    "organizaciones_sociales", "comunidades_indigenas", "vecinos",
    "colectivos_ambientales", "colectivos_especificos", "partidos_politicos",
    "policias_actores", "publico_general", "colectivos_violentos", "otros",
]

for col in COLS_ACTORES:
    if col not in df_chubut.columns:
        df_chubut[col] = 0
    if col not in df_jujuy.columns:
        df_jujuy[col] = 0

# accionar_policial solo está en Chubut → Jujuy queda en 99
if "accionar_policial" not in df_jujuy.columns:
    df_jujuy["accionar_policial"] = 99

# ============================================
# 4. CONCATENAR
# ============================================
df_chubut["provincia"] = "Chubut"
df_jujuy["provincia"] = "Jujuy"

# Columnas finales
COLS_FINALES = [
    "noticia_id", "provincia", "localidad",
    "actores_participantes", "demanda_principal",
    "tipo_accion", "tipo_accion_detalle",
    "disrupcion", "escala", "intensidad",
    "respuesta_estatal", "presencia_policial", "accionar_policial",
    "tipo_intervencion_estatal", "uso_fuerza", "heridos", "detenciones",
    "judicializacion", "coord_interinstitucional", "canales_informales",
    "amenaza_politica", "resultado_episodio",
    "fuente", "observaciones", "notas_estandarizacion",
] + COLS_ACTORES + [
    "demanda_salarial", "demanda_laboral", "demanda_ambiental_territorial",
    "demanda_politica_legal", "demanda_ddhh", "demanda_servicios",
    "demanda_derechos_especificos", "demanda_institucional", "demanda_conmemoracion",
]

# Dejar solo las finales
for col in COLS_FINALES:
    if col not in df_chubut.columns:
        df_chubut[col] = 0
    if col not in df_jujuy.columns:
        df_jujuy[col] = 0

df_chubut = df_chubut[COLS_FINALES]
df_jujuy = df_jujuy[COLS_FINALES]

df_merge = pd.concat([df_chubut, df_jujuy], ignore_index=True)

# ============================================
# 5. VERIFICACIÓN
# ============================================
print("=" * 60)
print("MERGE COMPLETADO")
print("=" * 60)
print(f"Total filas: {len(df_merge)}")
print(df_merge["provincia"].value_counts().to_string())
print(f"\nColumnas: {len(df_merge.columns)}")
print(list(df_merge.columns))

MERGE COMPLETADO
Total filas: 187
provincia
Chubut    121
Jujuy      66

Columnas: 47
['noticia_id', 'provincia', 'localidad', 'actores_participantes', 'demanda_principal', 'tipo_accion', 'tipo_accion_detalle', 'disrupcion', 'escala', 'intensidad', 'respuesta_estatal', 'presencia_policial', 'accionar_policial', 'tipo_intervencion_estatal', 'uso_fuerza', 'heridos', 'detenciones', 'judicializacion', 'coord_interinstitucional', 'canales_informales', 'amenaza_politica', 'resultado_episodio', 'fuente', 'observaciones', 'notas_estandarizacion', 'docentes', 'gremios_estatales', 'sindicatos_privados', 'organizaciones_sociales', 'comunidades_indigenas', 'vecinos', 'colectivos_ambientales', 'colectivos_especificos', 'partidos_politicos', 'policias_actores', 'publico_general', 'colectivos_violentos', 'otros', 'demanda_salarial', 'demanda_laboral', 'demanda_ambiental_territorial', 'demanda_politica_legal', 'demanda_ddhh', 'demanda_servicios', 'demanda_derechos_especificos', 'demanda_institucional'

In [22]:
df_merge

,noticia_id,provincia,localidad,actores_participantes,demanda_principal,tipo_accion,tipo_accion_detalle,disrupcion,escala,intensidad,...,otros,demanda_salarial,demanda_laboral,demanda_ambiental_territorial,demanda_politica_legal,demanda_ddhh,demanda_servicios,demanda_derechos_especificos,demanda_institucional,demanda_conmemoracion
0,2021103100,Chubut,Chubut,Sindicato de la Salud Pública,Pago de deuda de paritarias y retroactivos,Paro,trabajo a reglamento,1.0,media,media,...,0,1,0,0,0,0,0,0,0,0
1,2021110100,Chubut,No especificado,No especificado,No especificado,No especificado,No especificado,99.0,No especificado,No especificado,...,0,0,0,0,0,0,0,0,0,0
2,2021110200,Chubut,Rawson,Auxiliares de la educación,falta de convocatoria a paritarias salariales,Corte,Ruta provincial Nº 7,1.0,media,media,...,0,1,0,0,0,0,0,0,0,0
3,2021110202,Chubut,Esquel,Miembros de comunidades originarias,solución a la titularidad de tierras,Toma de edificio,ocupación del edificio del IAC,1.0,baja,media,...,0,0,0,1,0,0,0,0,0,0
4,2021110202,Chubut,Esquel,Miembros de comunidades originarias,No especificado,Toma de edificio,ocupación en la Secretaría de Bosques,1.0,baja,media,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,2023080101,Jujuy,CABA (fuera de Jujuy),Tercer Malón de la Paz,Contra Reforma / Por la Ley de Propiedad Comun...,Marcha,;ovilización / actividad de visibilización,0.0,Media,Media,...,0,0,0,1,1,0,0,0,0,0
183,2023080102,Jujuy,San Salvador de Jujuy,Docentes(secundarios y terciarios),Salarial,Mixta,movilización mixta,1.0,Baja,Baja,...,0,1,0,0,0,0,0,0,0,0
184,2023080301,Jujuy,San Salvador de Jujuy,Docentes/ Gremios docentes,Salarial/ Derecho a protesta,Mixta,movilización / paro,1.0,Media,Media,...,0,1,0,0,0,0,0,0,0,0
185,2023080501,Jujuy,San Salvador de Jujuy,La multisectorial,Contra Reforma/Salarial/ Apoyo Malón de la Paz,Marcha,marcha de antorchas,0.0,Alta,Alta,...,0,1,0,0,1,0,0,0,0,1


In [23]:
print(df_merge.groupby("provincia")["tipo_accion"].value_counts().to_string())

provincia  tipo_accion           
Chubut     Marcha                    30
           Manifestación             23
           Corte                     13
           Paro                      10
           Acto                      10
           No especificado            9
           Otra                       5
           Destrozos                  5
           Mixta                      4
           Escrache                   4
           Toma de edificio           3
           Asamblea                   3
           Caravana                   1
           Difusión de comunicado     1
Jujuy      Marcha                    30
           Corte                     14
           Mixta                     12
           Otra                       9
           Acampe                     1


In [24]:
def asignar_chubut(row):
    ta = str(row["tipo_accion"]).strip()
    det = str(row["tipo_accion_detalle"]).lower()
    
    row["tiene_marcha"] = 1 if ta in ["Marcha", "Manifestación", "Caravana"] else 0
    row["tiene_corte"] = 1 if ta == "Corte" else 0
    row["tiene_paro"] = 1 if ta == "Paro" else 0
    row["tiene_acto"] = 1 if ta in ["Acto", "Asamblea"] else 0
    row["tiene_escrache"] = 1 if ta == "Escrache" else 0
    row["tiene_destrozos"] = 1 if ta == "Destrozos" else 0
    row["tiene_toma"] = 1 if ta == "Toma de edificio" else 0
    
    return row

# Jujuy: mapeo con detalle
def asignar_jujuy(row):
    ta = str(row["tipo_accion"]).strip()
    det = str(row["tipo_accion_detalle"]).lower()
    
    row["tiene_marcha"] = 1 if (
        ta in ["Marcha"] or
        (ta == "Mixta" and ("marcha" in det or "movilizacion" in det)) or
        (ta == "Otra" and "marcha" in det)
    ) else 0
    
    row["tiene_corte"] = 1 if ta == "Corte" else 0
    
    row["tiene_paro"] = 1 if (
        (ta == "Otra" and "paro" in det) or
        (ta == "Mixta" and "paro" in det)
    ) else 0
    
    row["tiene_acto"] = 1 if (
        (ta == "Otra" and "asamblea" in det) or
        (ta == "Acampe" and "acampe" in det)
    ) else 0
    
    row["tiene_escrache"] = 0
    row["tiene_destrozos"] = 0
    row["tiene_toma"] = 0
    
    return row

# Aplicar
df_merge = df_merge.apply(lambda row: asignar_chubut(row) if row["provincia"] == "Chubut" else asignar_jujuy(row), axis=1)

In [25]:
import pandas as pd

COLS_ACCION = ["tiene_marcha", "tiene_corte", "tiene_paro", "tiene_acto",
               "tiene_escrache", "tiene_destrozos", "tiene_toma"]

# ============================================
# 1. CONTEOS ABSOLUTOS POR PROVINCIA
# ============================================
print("=" * 60)
print("CONTEOS ABSOLUTOS")
print("=" * 60)
conteos = df_merge.groupby("provincia")[COLS_ACCION].sum().T
print(conteos.to_string())

# ============================================
# 2. PORCENTAJES POR PROVINCIA
# ============================================
print("\n" + "=" * 60)
print("PORCENTAJES (% de eventos con cada tipo de acción)")
print("=" * 60)
porcentajes = df_merge.groupby("provincia")[COLS_ACCION].mean().T * 100
porcentajes = porcentajes.round(1)
print(porcentajes.to_string())

# ============================================
# 3. TABLA COMPARATIVA CON DIFERENCIA
# ============================================
print("\n" + "=" * 60)
print("COMPARACIÓN CON DIFERENCIA")
print("=" * 60)
comparacion = porcentajes.copy()
comparacion["diferencia_pp"] = comparacion["Jujuy"] - comparacion["Chubut"]
comparacion = comparacion.sort_values("diferencia_pp", ascending=False)
print(comparacion.to_string())

# ============================================
# 4. N POR PROVINCIA
# ============================================
print("\n" + "=" * 60)
print("TAMAÑO DE CADA CORPUS")
print("=" * 60)
print(df_merge["provincia"].value_counts().to_string())

# ============================================
# 5. GUARDAR
# ============================================


CONTEOS ABSOLUTOS
provincia        Chubut  Jujuy
tiene_marcha         54     36
tiene_corte          13     14
tiene_paro           10     14
tiene_acto           13      3
tiene_escrache        4      0
tiene_destrozos       5      0
tiene_toma            3      0

PORCENTAJES (% de eventos con cada tipo de acción)
provincia        Chubut  Jujuy
tiene_marcha       44.6   54.5
tiene_corte        10.7   21.2
tiene_paro          8.3   21.2
tiene_acto         10.7    4.5
tiene_escrache      3.3    0.0
tiene_destrozos     4.1    0.0
tiene_toma          2.5    0.0

COMPARACIÓN CON DIFERENCIA
provincia        Chubut  Jujuy  diferencia_pp
tiene_paro          8.3   21.2           12.9
tiene_corte        10.7   21.2           10.5
tiene_marcha       44.6   54.5            9.9
tiene_toma          2.5    0.0           -2.5
tiene_escrache      3.3    0.0           -3.3
tiene_destrozos     4.1    0.0           -4.1
tiene_acto         10.7    4.5           -6.2

TAMAÑO DE CADA CORPUS
provincia
Chubu

In [26]:
df_merge.to_excel("../data/processed/eventos_chubut_jujuy.xlsx")